In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from torch.utils.data import DataLoader

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
batch_size = 128
image_size = 28
flat_dim = image_size * image_size
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    num_workers=0,
)
print(f"MNIST training samples: {len(train_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")

In [ ]:
class WeightedGenerator(nn.Module):
    def __init__(self, latent_dim=100, output_dim=784):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.value_layer = nn.Linear(1024, output_dim)
        self.weight_layer = nn.Linear(1024, output_dim)
    def forward(self, z):
        h = self.backbone(z)
        value = torch.tanh(self.value_layer(h))
        weight = torch.sigmoid(self.weight_layer(h))
        weighted_value = weight * value
        out = weighted_value.view(z.size(0), 1, 28, 28)
        return out, value, weight, weighted_value
class LinearDiscriminator(nn.Module):
    def __init__(self, input_dim=784):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )
    def forward(self, x):
        x = x.view(x.size(0), -1)
        return torch.sigmoid(self.model(x))

In [ ]:
latent_dim = 100
generator = WeightedGenerator(latent_dim=latent_dim, output_dim=flat_dim).to(device)
discriminator = LinearDiscriminator(input_dim=flat_dim).to(device)
lr_g = 2e-4
lr_d = 2e-4
epochs = 25 if torch.cuda.is_available() else 10
eps = 1e-8
optimizer_g = torch.optim.Adam(generator.parameters(), lr=lr_g, betas=(0.5, 0.999))
optimizer_d = torch.optim.Adam(discriminator.parameters(), lr=lr_d, betas=(0.5, 0.999))
history_d = []
history_g = []
fixed_noise = torch.randn(64, latent_dim, device=device)
for epoch in range(1, epochs + 1):
    epoch_d_loss = 0.0
    epoch_g_loss = 0.0
    for real_batch, _ in train_loader:
        real_batch = real_batch.to(device)
        current_batch = real_batch.size(0)
        z = torch.randn(current_batch, latent_dim, device=device)
        fake_batch, _, _, _ = generator(z)
        d_real = discriminator(real_batch)
        d_fake = discriminator(fake_batch.detach())
        d_loss = -torch.sum(torch.log(d_real + eps)) - torch.sum(torch.log(1.0 - d_fake + eps))
        optimizer_d.zero_grad(set_to_none=True)
        d_loss.backward()
        optimizer_d.step()
        z_g = torch.randn(current_batch, latent_dim, device=device)
        fake_for_g, _, _, _ = generator(z_g)
        d_fake_for_g = discriminator(fake_for_g)
        g_loss = -torch.sum(torch.log(d_fake_for_g + eps))
        optimizer_g.zero_grad(set_to_none=True)
        g_loss.backward()
        optimizer_g.step()
        epoch_d_loss += d_loss.item() / current_batch
        epoch_g_loss += g_loss.item() / current_batch
    mean_d = epoch_d_loss / len(train_loader)
    mean_g = epoch_g_loss / len(train_loader)
    history_d.append(mean_d)
    history_g.append(mean_g)
    if epoch == 1 or epoch % max(1, epochs // 10) == 0:
print("Training complete.")

In [ ]:
with torch.no_grad():
    generated_images, generated_values, generated_weights, weighted_values = generator(fixed_noise)

generated_images_display = ((generated_images + 1.0) / 2.0).clamp(0.0, 1.0)

real_batch_vis, _ = next(iter(train_loader))
real_batch_vis = real_batch_vis[:64]
real_images_display = ((real_batch_vis + 1.0) / 2.0).clamp(0.0, 1.0)

gen_grid = make_grid(generated_images_display[:64].cpu(), nrow=8, padding=2)
real_grid = make_grid(real_images_display.cpu(), nrow=8, padding=2)

plt.figure(figsize=(16, 5))

plt.subplot(1, 3, 1)
plt.plot(history_d, label="Discriminator loss")
plt.plot(history_g, label="Generator loss")
plt.xlabel("Epoch")
plt.ylabel("Average sum-loss per sample")
plt.title("Training Loss Curves")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 3, 2)
plt.imshow(np.transpose(real_grid.numpy(), (1, 2, 0)), cmap="gray")
plt.title("Real MNIST Samples")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(np.transpose(gen_grid.numpy(), (1, 2, 0)), cmap="gray")
plt.title("Generated Samples")
plt.axis("off")

plt.tight_layout()
plt.show()

real_comp = real_images_display.to(device)
gen_comp = generated_images_display[: real_comp.size(0)]
indexwise_mse = torch.mean((gen_comp - real_comp) ** 2).item()
gen_flat = gen_comp.view(gen_comp.size(0), -1)
real_flat = real_comp.view(real_comp.size(0), -1)
pairwise_sq_l2 = torch.cdist(gen_flat, real_flat, p=2) ** 2
nearest_real_mse = (pairwise_sq_l2.min(dim=1).values / gen_flat.size(1)).mean().item()
mean_intensity_gap = torch.abs(gen_comp.mean() - real_comp.mean()).item()

print(f"Mean generated weight: {generated_weights.mean().item():.6f}")
print(f"Std generated weight: {generated_weights.std().item():.6f}")
print(f"Mean absolute intensity gap (generated vs real): {mean_intensity_gap:.6f}")
print(f"Index-wise MSE (generated[i] vs real[i]): {indexwise_mse:.6f}")
print(f"Nearest-real MSE (each generated image to closest real image in batch): {nearest_real_mse:.6f}")
print(f"Generated tensor shape: {tuple(generated_images.shape)}")